In [87]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
openai_client = OpenAI()

In [88]:
def rag(self, query):
    search_results = self.search(query)
    prompt = self.build_prompt(query, search_results)
    answer = self.llm(prompt)
    return answer

In [89]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Yes—usually you can, as long as the course is still open for enrollment and you meet any prerequisites.\n\nIf you want, I can help you figure out the next step. For example, I can draft a short message asking:\n- whether late enrollment is still possible,\n- how to register,\n- and whether you’re missing anything needed to join.\n\nIf you’re asking about a specific course, send me the course name or details and I’ll help you check what to say.'

In [90]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [91]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [92]:
from rag_helper import RAGBase

instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [93]:
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [94]:
search_tool = {
    "type": "function",
    'name': 'search',
    'description': 'Search the FAQ database for entries matching the given query.',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                "type": "string",
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        "required": ["query"],
        'additionalProperties': False
    }
}

In [95]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"Can I join late discovered course enrollment join after start FAQ"}', call_id='call_C6tIqVBemPXTq0Zs3kODFHOr', name='search', type='function_call', id='fc_03c42d1fc30ec7d8006a38354bf37c819e9f5451b925e489c2', namespace=None, status='completed')]

In [96]:
len(response.output)

1

In [97]:
import json

call = response.output[0]
call

ResponseFunctionToolCall(arguments='{"query":"Can I join late discovered course enrollment join after start FAQ"}', call_id='call_C6tIqVBemPXTq0Zs3kODFHOr', name='search', type='function_call', id='fc_03c42d1fc30ec7d8006a38354bf37c819e9f5451b925e489c2', namespace=None, status='completed')

In [98]:
import json
args = json.loads(call.arguments)
args


{'query': 'Can I join late discovered course enrollment join after start FAQ'}

In [99]:
call.name

'search'

In [100]:
results = search(**args)

In [101]:
result_json = json.dumps(results, indent=2)
result_json

'[\n  {\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions.",\n    "doc_id": "74eb249bbf"\n  },\n  {\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "How should I start the course and follow the weekly workflow?",\n    "answer": "Start with the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp).\\n\\nYou can start whenever you want. The videos and GitHub materials are available, and the deadlines are listed in the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).\\n\

In [102]:
function_call_output = {
    "type": "function_call_output",
    'call_id': call.call_id,
    'output': result_json,
}

In [103]:
messages.append(call)

In [104]:
messages.append(function_call_output)

In [105]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"Can I join late discovered course enrollment join after start FAQ"}', call_id='call_C6tIqVBemPXTq0Zs3kODFHOr', name='search', type='function_call', id='fc_03c42d1fc30ec7d8006a38354bf37c819e9f5451b925e489c2', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_C6tIqVBemPXTq0Zs3kODFHOr',
  'output': '[\n  {\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions.",\n    "doc_id": "74eb249bbf"\n  },\n  {\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "How should I start the course and follow the weekly workflow?",\n    "answer": "Start with the 

In [106]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [107]:
print(response.output_text)

Yes — you can still join and start learning anytime.

If you want a certificate, though, you need to submit your project while the course is still accepting submissions.


In [108]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(777, 37)

In [109]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    # Prices per 1M tokens (example pricing)
    INPUT_PRICE_PER_MILLION = 0.15   # $0.15 / 1M input tokens
    OUTPUT_PRICE_PER_MILLION = 0.60  # $0.60 / 1M output tokens

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION

    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost
    }


# Your tokens
result = calculate_gpt54mini_price(652, 33)

print("Total Cost: $", round(result["total_cost"], 8))

Total Cost: $ 0.0001176


In [110]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == 'search':
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        'call_id': call.call_id,
        'output': result_json,
    }

In [111]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'


messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

In [112]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [113]:
response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enroll registration late join FAQ"}', call_id='call_wmkaWYjs64dQ9xDZOWNTcb6R', name='search', type='function_call', id='fc_0bfcbaf59d9f8748006a38354eadf8819d965e16abf0c6ca02', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"new student discovered course can I still join course FAQ"}', call_id='call_qIjmlJR5cHrkSHeeBBHsNtsF', name='search', type='function_call', id='fc_0bfcbaf59d9f8748006a38354eae08819d9a34c6542d15623f', namespace=None, status='completed')]

In [114]:
messages.extend(response.output)

for item in response.output:
    if item.type == 'function_call':
        print('function_call:', item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)

    elif item.type == 'message':
        print('ASSISTANT:')
        print(item.content[0].text)

function_call: search {"query":"join course discovered course can I join enroll registration late join FAQ"}
function_call: search {"query":"new student discovered course can I still join course FAQ"}


In [115]:
messages

[{'role': 'developer',
  'content': "\nYou're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches.\n\nTry to expand your search by using new keywords\nbased on the results you get from the search.\n\nAt the end, ask if there are other areas that the user wants to explore.\n"},
 {'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"join course discovered course can I join enroll registration late join FAQ"}', call_id='call_wmkaWYjs64dQ9xDZOWNTcb6R', name='search', type='function_call', id='fc_0bfcbaf59d9f8748006a38354eadf8819d965e16abf0c6ca02', namespace=None, status='completed'),
 ResponseFunctionToolCall(arguments='{"query":"new student discovered course can I still join course FAQ"}', cal

In [116]:
messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

it = 1

while True:
    print(f'iteration #{it}...')
    has_function_calls = False

    response = openai_client.responses.create(
        model='gpt-5.4-mini',
        input=messages,
        tools=[search_tool]
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == 'function_call':
            print('function_call:', item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == 'message':
            print('ASSISTANT:')
            print(item.content[0].text)
    
    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
function_call: search {"query":"join course discovered course can I join enroll registration faq"}
function_call: search {"query":"new student join the course late enrollment FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

If you want a certificate, though, you need to submit your project while submissions are still open. Otherwise, you can still follow the materials and learn at your own pace.

If you'd like, I can also help you figure out how to start the course or what the weekly workflow looks like.


In [117]:
def agent_loop(instructions, question, model='gpt-5.4-mini') -> str:
    messages = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': question}
    ]

    it = 1

    while True:
        print(f'iteration #{it}...')
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == 'function_call':
                print('function_call:', item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == 'message':
                print('ASSISTANT:')
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break
    
    return last_answer

In [118]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'

In [119]:
result = agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"join course late enrollment discovered course can I join"}
function_call: search {"query":"enrollment open join course discovered the course can I join"}
function_call: search {"query":"course FAQ join later discovered course registration enrollment"}
iteration #2...
ASSISTANT:
Yes — you can still join the course even if you just discovered it.

A couple of important notes:
- You can start learning and submit homework while the submission form is open.
- If you want a certificate, you’ll need to submit your project while submissions are still being accepted.
- Certificates are only available if you finish with the live cohort, not in self-paced mode.

If you want, I can also explain how to get started or how the weekly workflow works.


In [120]:
result

'Yes — you can still join the course even if you just discovered it.\n\nA couple of important notes:\n- You can start learning and submit homework while the submission form is open.\n- If you want a certificate, you’ll need to submit your project while submissions are still being accepted.\n- Certificates are only available if you finish with the live cohort, not in self-paced mode.\n\nIf you want, I can also explain how to get started or how the weekly workflow works.'

In [121]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
"""

question = "what's queen gambit?"

result = agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"queen's gambit opening chess course FAQ"}
iteration #3...
ASSISTANT:
I couldn’t find a course FAQ entry for “queen gambit,” so it’s likely not a course/logistics topic.

If you meant **“Queen’s Gambit”** in chess, I can’t answer that from the course FAQ database. If you meant something else related to the course, please rephrase and I’ll check again.

Is there another course-related area you want to explore?
